In [2]:
import pandas as pd
# from main import SatCLIPLightningModule
# import location_encoder as LE
# from temporal_encoding import Fourier, Direct
from tqdm import tqdm
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
import numpy as np

from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import io
from PIL import Image

from utils import *
import torch.nn.functional as F

In [3]:
# Load the GHCN dataset
ghcn_df = pd.read_csv('ghcn_1992_2024_temp_and_prec.csv')
print(len(ghcn_df))
ghcn_df.head()

688037392


,ID,Date,Element,Value,MFlag,QFlag,SFlag,OBS-TIME,Latitude,Longitude,Elevation,Name
0,ASN00010725,19920101,PRCP,0,NaN,NaN,a,NaN,-33.4458,117.6614,274.0,CONGEE
1,ASN00010729,19920101,PRCP,0,NaN,NaN,a,NaN,-34.3439,118.6306,104.0,CHILLINUP
2,ASN00010776,19920101,PRCP,0,NaN,NaN,a,NaN,-32.2072,117.4042,300.0,SOUTH CAROLING
3,ASN00010779,19920101,PRCP,0,NaN,NaN,a,NaN,-34.0042,118.0536,240.0,MIANELUP
4,ASN00010649,19920101,PRCP,0,NaN,NaN,a,NaN,-33.5253,117.6456,300.0,WAYVILLE


In [ ]:
# Filter GHCN to quality controlled data and limit to TMAX (maximum temperature) and TMIN (minimum temperature) observations
ghcn_qc_temp_df = ghcn_df[((ghcn_df["Element"]=="TMAX") & (ghcn_df["QFlag"].isna())) | 
                          (ghcn_df["Element"]=="TMIN") & (ghcn_df["QFlag"].isna())].copy()
ghcn_qc_temp_df['Date'] = pd.to_datetime(ghcn_qc_temp_df['Date'], format="%Y%m%d")
ghcn_qc_temp_df.loc[:, 'DayOfYear'] = (ghcn_qc_temp_df['Date'].dt.dayofyear - 1) / 364.0
ghcn_qc_temp_df.loc[:, 'POSIX_TIME'] = ghcn_qc_temp_df['Date'].astype(np.int64) // 10**9
ghcn_qc_temp_df.head()

,ID,Date,Element,Value,MFlag,QFlag,SFlag,OBS-TIME,Latitude,Longitude,Elevation,Name,DayOfYear,POSIX_TIME
35,ASN00010582,1992-01-01,TMAX,223,NaN,NaN,a,NaN,-33.8300,117.1590,305.0,KOJONUP,0.0,694224000
36,ASN00010582,1992-01-01,TMIN,144,NaN,NaN,a,NaN,-33.8300,117.1590,305.0,KOJONUP,0.0,694224000
44,ASN00010592,1992-01-01,TMAX,264,NaN,NaN,a,NaN,-33.1006,118.4625,286.0,LAKE GRACE COMPARISON,0.0,694224000
45,ASN00010592,1992-01-01,TMIN,147,NaN,NaN,a,NaN,-33.1006,118.4625,286.0,LAKE GRACE COMPARISON,0.0,694224000
58,ASN00010612,1992-01-01,TMAX,280,NaN,NaN,a,NaN,-32.0656,118.3956,276.0,NAREMBEEN,0.0,694224000


In [15]:
ghcnd_benchmark_temp = ghcn_qc_temp_df.pivot_table(
    index=["ID", "Date", "Name", "Latitude", "Longitude", "Elevation"],
    columns="Element",
    values="Value",
    aggfunc="first",   # or "mean" if there might be duplicates
).reset_index()

ghcnd_benchmark_temp.columns.name = None

In [16]:
# Filter out rows where either TMAX or TMIN is NaN
ghcnd_benchmark_temp = ghcnd_benchmark_temp.dropna(subset=['TMAX', 'TMIN'])
ghcnd_benchmark_temp['TMAX'] = ghcnd_benchmark_temp['TMAX'] / 10.0  # Convert to degrees Celsius
ghcnd_benchmark_temp['TMIN'] = ghcnd_benchmark_temp['TMIN'] / 10.0  # Convert to degrees Celsius
ghcnd_benchmark_temp['TMEAN'] = (ghcnd_benchmark_temp['TMAX'] + ghcnd_benchmark_temp['TMIN']) / 2  # Convert to degrees Celsius
ghcnd_benchmark_temp.head()

,ID,Date,Name,Latitude,Longitude,Elevation,TMAX,TMIN,TMEAN
0,AE000041196,1992-01-01,SHARJAH INTER. AIRP,25.333,55.517,34.0,26.9,9.7,18.30
1,AE000041196,1992-01-02,SHARJAH INTER. AIRP,25.333,55.517,34.0,28.4,14.0,21.20
2,AE000041196,1992-01-03,SHARJAH INTER. AIRP,25.333,55.517,34.0,26.4,18.3,22.35
7,AE000041196,1992-01-08,SHARJAH INTER. AIRP,25.333,55.517,34.0,20.6,9.7,15.15
8,AE000041196,1992-01-10,SHARJAH INTER. AIRP,25.333,55.517,34.0,21.9,5.9,13.90


In [19]:
ghcnd_benchmark_temp["Date"].max()

Timestamp('2024-12-31 00:00:00')

In [20]:
ghcnd_benchmark_temp.to_csv('../data/ghcnd_1992_2024_temp_benchmark.csv', index=False, header=False)
ghcnd_benchmark_temp.to_parquet('../data/ghcnd_1992_2024_temp_benchmark.parquet', index=False, engine='pyarrow')

In [21]:
print("Num_observations: ", len(ghcnd_benchmark_temp))
print("Num_stations: ", len(ghcnd_benchmark_temp.groupby(['Latitude', 'Longitude'])))

Num_observations:  169763637
Num_stations:  26174


In [ ]:
# Plot the splits on a map
fig, ax = plt.subplots(figsize=(12, 6))
m = Basemap(projection='cyl', resolution='c', ax=ax)
m.drawcoastlines()
# Parallels (Latitudes) range from -90 to 90
m.drawparallels(np.arange(-90, 90, 30), labels=[True, True, False, False])

# Meridians (Longitudes) range from 0 to 360 (or -180 to 180 depending on projection)
m.drawmeridians(np.arange(0, 360, 30), labels=[False, False, True, True])

sc = ax.scatter(ghcnd_benchmark_temp["Latitude"], ghcnd_benchmark_temp["Longitude"], c=ghcnd_benchmark_temp["TMEAN"], cmap='coolwarm', marker='o', alpha=0.5)
fig.colorbar(sc, label='Split', ax=ax)